# Retail Price Optimization Capstone

## 03 - Statistical Analysis and Preliminary Models

**Student:** Bhupesh Patni  
**Stage:** Graded Interim Report

### Purpose

This notebook performs reproducible sample-size calculations, formal statistical analysis for RQ1 and RQ2, an exploratory interaction analysis for RQ4, and preliminary chronological demand-model comparisons for RQ3.

### Interpretation boundaries

- The demand outcome is `qty`.
- `unit_price` is a pricing decision variable and predictor.
- Results are associational because the dataset is observational.
- The test partition remains untouched during preliminary model selection.
- Any later optimization will concern simulated revenue, not profit or proven business uplift.


## 1. Setup and package check


In [ ]:
from pathlib import Path
import json
import platform
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import ncf, f
from statsmodels.stats.oneway import anova_oneway
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")

print(f"Python: {sys.version.split()[0]}")
print(f"Interpreter: {sys.executable}")
print(f"Platform: {platform.platform()}")
print("All required imports completed successfully.")


## 2. Locate the project and load the analysis-ready data


In [ ]:
current_dir = Path.cwd().resolve()

if current_dir.name.lower() == "notebooks":
    PROJECT_ROOT = current_dir.parent
elif (current_dir / "data").exists() and (current_dir / "notebooks").exists():
    PROJECT_ROOT = current_dir
else:
    candidates = [current_dir, *current_dir.parents]
    PROJECT_ROOT = next(
        (
            path for path in candidates
            if (path / "data").exists() and (path / "notebooks").exists()
        ),
        current_dir,
    )

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
TABLES_DIR = PROJECT_ROOT / "reports" / "tables"

for folder in [PROCESSED_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DATA_PATH = PROCESSED_DIR / "retail_price_analysis_ready.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing file: {DATA_PATH}. "
        "Run 02_eda_and_preprocessing_updated.ipynb first."
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Input file: {DATA_PATH}")


In [ ]:
df = pd.read_csv(DATA_PATH)
df["month_year"] = pd.to_datetime(
    df["month_year"],
    format="%Y-%m-%d",
    errors="coerce",
)

required_columns = {
    "product_id", "product_category_name", "month_year", "qty",
    "unit_price", "freight_price", "product_score", "avg_comp_price",
    "relative_price", "competitive_price_position", "previous_qty",
    "previous_unit_price", "rolling_qty_mean_3", "rolling_price_mean_3",
    "calendar_month", "calendar_year", "data_partition",
}

missing_columns = sorted(required_columns.difference(df.columns))
if missing_columns:
    raise KeyError("Missing required columns: " + ", ".join(missing_columns))

if df["month_year"].isna().any():
    raise ValueError("Invalid month_year values were found.")

expected_start = pd.Timestamp("2017-01-01")
expected_end = pd.Timestamp("2018-08-01")
if df["month_year"].min() != expected_start or df["month_year"].max() != expected_end:
    raise ValueError(
        "Unexpected date range. Expected 2017-01-01 through 2018-08-01."
    )

partition_counts = (
    df["data_partition"]
    .value_counts()
    .reindex(["Train", "Validation", "Test"])
)

if partition_counts.isna().any() or (partition_counts <= 0).any():
    raise ValueError("Train, Validation, and Test must all be non-empty.")

print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
print(f"Date range: {df['month_year'].min():%Y-%m-%d} to {df['month_year'].max():%Y-%m-%d}")
display(partition_counts.rename("Rows").to_frame())


## 3. Reproducible sample-size and sensitivity analysis

RQ1 uses one-way ANOVA power. RQ2 uses a noncentral-F calculation for the global multiple-regression test. This avoids parameter-name ambiguity in older power-analysis interfaces. RQ3 uses chronological learning curves, while RQ4 remains exploratory because of sparse cells.


In [ ]:
ALPHA = 0.05
TARGET_POWER = 0.80
NUM_CATEGORIES = int(df["product_category_name"].nunique())
NUM_RQ2_PREDICTORS = 8


def anova_power(total_n, groups, effect_f, alpha=0.05):
    """Approximate balanced one-way ANOVA power using a noncentral F."""
    df_num = groups - 1
    df_den = total_n - groups
    if df_den <= 0:
        return np.nan
    critical = f.ppf(1 - alpha, df_num, df_den)
    noncentrality = total_n * (effect_f ** 2)
    return 1 - ncf.cdf(critical, df_num, df_den, noncentrality)


def regression_power(total_n, predictors, effect_f2, alpha=0.05):
    """Power for the global multiple-regression F test using Cohen's f-squared."""
    df_num = predictors
    df_den = total_n - predictors - 1
    if df_den <= 0:
        return np.nan
    critical = f.ppf(1 - alpha, df_num, df_den)
    noncentrality = effect_f2 * total_n
    return 1 - ncf.cdf(critical, df_num, df_den, noncentrality)


def minimum_n(power_function, target_power=0.80, start_n=10, max_n=20000):
    """Find the smallest integer N meeting the target power."""
    for n_value in range(start_n, max_n + 1):
        power_value = power_function(n_value)
        if np.isfinite(power_value) and power_value >= target_power:
            return n_value
    return np.nan


sample_rows = []

# RQ1 sensitivity scenarios.
for label, effect_f in [("Small", 0.10), ("Medium", 0.25), ("Large", 0.40)]:
    required_n = minimum_n(
        lambda n: anova_power(n, NUM_CATEGORIES, effect_f, ALPHA),
        target_power=TARGET_POWER,
        start_n=NUM_CATEGORIES + 2,
    )
    achieved = anova_power(len(df), NUM_CATEGORIES, effect_f, ALPHA)
    sample_rows.append({
        "RQ": "RQ1",
        "Method": "Balanced one-way ANOVA sensitivity",
        "Assumption": label,
        "Effect_Size": effect_f,
        "Effect_Size_Metric": "Cohen f",
        "Alpha": ALPHA,
        "Target_Power": TARGET_POWER,
        "Required_N": required_n,
        "Actual_N": len(df),
        "Nominal_Achieved_Power": achieved,
    })

# RQ2 sensitivity scenarios.
for label, effect_f2 in [("Small", 0.02), ("Medium", 0.15), ("Large", 0.35)]:
    required_n = minimum_n(
        lambda n: regression_power(n, NUM_RQ2_PREDICTORS, effect_f2, ALPHA),
        target_power=TARGET_POWER,
        start_n=NUM_RQ2_PREDICTORS + 3,
    )
    achieved = regression_power(
        len(df), NUM_RQ2_PREDICTORS, effect_f2, ALPHA
    )
    sample_rows.append({
        "RQ": "RQ2",
        "Method": "Multiple-regression global F sensitivity",
        "Assumption": label,
        "Effect_Size": effect_f2,
        "Effect_Size_Metric": "Cohen f-squared",
        "Alpha": ALPHA,
        "Target_Power": TARGET_POWER,
        "Required_N": required_n,
        "Actual_N": len(df),
        "Nominal_Achieved_Power": achieved,
    })

sample_rows.extend([
    {
        "RQ": "RQ3",
        "Method": "Chronological learning curve and validation stability",
        "Assumption": "Predictive adequacy",
        "Effect_Size": np.nan,
        "Effect_Size_Metric": "Not applicable",
        "Alpha": np.nan,
        "Target_Power": np.nan,
        "Required_N": np.nan,
        "Actual_N": len(df),
        "Nominal_Achieved_Power": np.nan,
    },
    {
        "RQ": "RQ4",
        "Method": "Exploratory interaction with observed cell-count audit",
        "Assumption": "Sparse and unbalanced cells",
        "Effect_Size": np.nan,
        "Effect_Size_Metric": "Not estimated",
        "Alpha": ALPHA,
        "Target_Power": TARGET_POWER,
        "Required_N": np.nan,
        "Actual_N": len(df),
        "Nominal_Achieved_Power": np.nan,
    },
])

sample_size_results = pd.DataFrame(sample_rows)
display(sample_size_results)

sample_size_results.to_csv(
    TABLES_DIR / "table_17_sample_size_analysis.csv",
    index=False,
)

print("Sample-size and sensitivity calculations completed successfully.")


### Sample-size limitation

The power values are nominal because classical power calculations assume independent observations. The dataset contains repeated product-month records. Product-clustered inference, chronological validation, sensitivity analysis, and transparent limitations are therefore used instead of treating all 676 rows as fully independent.


## 4. RQ1: Relative price differences across categories


In [ ]:
rq1_groups = [
    group["relative_price"].dropna().to_numpy()
    for _, group in df.groupby("product_category_name")
]

levene_stat, levene_p = stats.levene(*rq1_groups, center="median")
welch_result = anova_oneway(
    rq1_groups,
    use_var="unequal",
    welch_correction=True,
)
kruskal_stat, kruskal_p = stats.kruskal(*rq1_groups)

grand_mean = df["relative_price"].mean()
ss_between = sum(
    len(group) * (group["relative_price"].mean() - grand_mean) ** 2
    for _, group in df.groupby("product_category_name")
)
ss_total = ((df["relative_price"] - grand_mean) ** 2).sum()
eta_squared = ss_between / ss_total

rq1_results = pd.DataFrame({
    "Test": ["Levene median", "Welch ANOVA", "Kruskal-Wallis", "Eta-squared"],
    "Statistic": [
        levene_stat,
        welch_result.statistic,
        kruskal_stat,
        eta_squared,
    ],
    "p_value": [
        levene_p,
        welch_result.pvalue,
        kruskal_p,
        np.nan,
    ],
    "Interpretation_Role": [
        "Variance diagnostic",
        "Primary omnibus comparison",
        "Robust sensitivity comparison",
        "Descriptive effect size",
    ],
})

display(rq1_results)
rq1_results.to_csv(
    TABLES_DIR / "table_18_rq1_statistical_results.csv",
    index=False,
)


In [ ]:
categories = sorted(df["product_category_name"].dropna().unique())
pairwise_rows = []

for index, category_a in enumerate(categories):
    for category_b in categories[index + 1:]:
        values_a = df.loc[
            df["product_category_name"] == category_a,
            "relative_price",
        ]
        values_b = df.loc[
            df["product_category_name"] == category_b,
            "relative_price",
        ]
        statistic, raw_p = stats.mannwhitneyu(
            values_a,
            values_b,
            alternative="two-sided",
        )
        pairwise_rows.append({
            "Category_A": category_a,
            "Category_B": category_b,
            "U_Statistic": statistic,
            "Raw_p": raw_p,
        })

rq1_posthoc = pd.DataFrame(pairwise_rows)
rq1_posthoc["Holm_Adjusted_p"] = multipletests(
    rq1_posthoc["Raw_p"],
    method="holm",
)[1]
rq1_posthoc["Significant_at_0.05"] = (
    rq1_posthoc["Holm_Adjusted_p"] < 0.05
)
rq1_posthoc = rq1_posthoc.sort_values("Holm_Adjusted_p")

display(rq1_posthoc.head(15))
rq1_posthoc.to_csv(
    TABLES_DIR / "table_18b_rq1_posthoc.csv",
    index=False,
)


In [ ]:
category_order = (
    df.groupby("product_category_name")["relative_price"]
    .median()
    .sort_values()
    .index
)

fig, ax = plt.subplots(figsize=(11, 6))
sns.boxplot(
    data=df,
    x="relative_price",
    y="product_category_name",
    order=category_order,
    color="#4C78A8",
    ax=ax,
)
ax.axvline(1.0, color="#D62728", linestyle="--", linewidth=1.4)
ax.set_title("RQ1: Relative Price by Product Category")
ax.set_xlabel("Relative price: unit price / average competitor price")
ax.set_ylabel("Product category")
fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "figure_06_rq1_category_comparison.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## 5. RQ2: Interpretable demand-association model

Quantity, unit price, and average competitor price are log transformed. Freight price uses `log1p` because zero freight is valid and the direct logarithm of zero is undefined. Product-clustered standard errors account for repeated monthly observations. The unit-price coefficient is an association-based elasticity estimate, not a causal effect.


In [ ]:
reg_df = df.copy()

for column in ["qty", "unit_price", "avg_comp_price"]:
    nonpositive_count = int((reg_df[column] <= 0).sum())
    if nonpositive_count > 0:
        raise ValueError(
            f"{column} contains {nonpositive_count} nonpositive values."
        )

negative_freight_count = int((reg_df["freight_price"] < 0).sum())
if negative_freight_count > 0:
    raise ValueError(
        f"freight_price contains {negative_freight_count} negative values."
    )

zero_freight_count = int((reg_df["freight_price"] == 0).sum())
print(f"Zero freight-price rows retained: {zero_freight_count:,}")

reg_df["log_qty"] = np.log(reg_df["qty"])
reg_df["log_unit_price"] = np.log(reg_df["unit_price"])
reg_df["log_avg_comp_price"] = np.log(reg_df["avg_comp_price"])
reg_df["log1p_freight_price"] = np.log1p(reg_df["freight_price"])

transformed_columns = [
    "log_qty",
    "log_unit_price",
    "log_avg_comp_price",
    "log1p_freight_price",
]

invalid_transforms = (
    reg_df[transformed_columns]
    .replace([np.inf, -np.inf], np.nan)
    .isna()
    .sum()
)

if invalid_transforms.sum() > 0:
    display(invalid_transforms.rename("Invalid_Count").to_frame())
    raise ValueError("Invalid values were found after transformation.")

rq2_formula = (
    "log_qty ~ log_unit_price + log_avg_comp_price + "
    "log1p_freight_price + product_score + calendar_month + "
    "C(calendar_year) + C(product_category_name)"
)

rq2_ols = smf.ols(
    formula=rq2_formula,
    data=reg_df,
).fit()

rq2_clustered = rq2_ols.get_robustcov_results(
    cov_type="cluster",
    groups=reg_df["product_id"],
)

confidence_intervals = rq2_clustered.conf_int()

rq2_table = pd.DataFrame({
    "Term": rq2_ols.params.index,
    "Coefficient": rq2_clustered.params,
    "Clustered_SE": rq2_clustered.bse,
    "t_value": rq2_clustered.tvalues,
    "p_value": rq2_clustered.pvalues,
    "CI_2.5%": confidence_intervals[:, 0],
    "CI_97.5%": confidence_intervals[:, 1],
})

display(rq2_table)
print(f"Observations: {int(rq2_ols.nobs):,}")
print(f"Product clusters: {reg_df['product_id'].nunique():,}")
print(f"R-squared: {rq2_ols.rsquared:.4f}")
print(f"Adjusted R-squared: {rq2_ols.rsquared_adj:.4f}")

rq2_table.to_csv(
    TABLES_DIR / "table_19_rq2_regression_results.csv",
    index=False,
)

print("RQ2 regression completed successfully.")


In [ ]:
fitted_values = rq2_ols.fittedvalues
residuals = rq2_ols.resid
standardized_residuals = residuals / residuals.std(ddof=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.scatterplot(
    x=fitted_values,
    y=residuals,
    alpha=0.65,
    ax=axes[0],
)
axes[0].axhline(0, color="#D62728", linestyle="--")
axes[0].set_title("RQ2 Residuals versus Fitted Values")
axes[0].set_xlabel("Fitted log quantity")
axes[0].set_ylabel("Residual")

stats.probplot(
    standardized_residuals,
    dist="norm",
    plot=axes[1],
)
axes[1].set_title("RQ2 Normal Q-Q Plot")

fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "figure_07_regression_diagnostics.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## 6. RQ4: Exploratory category × competitive-price-position interaction


In [ ]:
cell_counts = pd.crosstab(
    df["product_category_name"],
    df["competitive_price_position"],
)

sparse_cells = int((cell_counts < 5).sum().sum())
print(f"Cells with fewer than 5 observations: {sparse_cells}")
display(cell_counts)

rq4_formula = (
    "np.log1p(qty) ~ C(product_category_name) * "
    "C(competitive_price_position) + calendar_month + C(calendar_year)"
)

rq4_ols = smf.ols(
    formula=rq4_formula,
    data=df,
).fit()

rq4_clustered = rq4_ols.get_robustcov_results(
    cov_type="cluster",
    groups=df["product_id"],
)

rq4_confidence = rq4_clustered.conf_int()

rq4_table = pd.DataFrame({
    "Term": rq4_ols.params.index,
    "Coefficient": rq4_clustered.params,
    "Clustered_SE": rq4_clustered.bse,
    "p_value": rq4_clustered.pvalues,
    "CI_2.5%": rq4_confidence[:, 0],
    "CI_97.5%": rq4_confidence[:, 1],
})

rq4_table["Exploratory_Due_to_Sparse_Cells"] = sparse_cells > 0

display(rq4_table)
rq4_table.to_csv(
    TABLES_DIR / "table_21_rq4_interaction_results.csv",
    index=False,
)


## 7. RQ3: Preliminary demand models with chronological validation


In [ ]:
train_df = df[df["data_partition"] == "Train"].copy()
validation_df = df[df["data_partition"] == "Validation"].copy()
test_df = df[df["data_partition"] == "Test"].copy()

TARGET = "qty"
NUMERIC_FEATURES = [
    "unit_price",
    "avg_comp_price",
    "freight_price",
    "product_score",
    "calendar_month",
    "calendar_year",
    "previous_qty",
    "previous_unit_price",
    "rolling_qty_mean_3",
    "rolling_price_mean_3",
]
CATEGORICAL_FEATURES = [
    "product_category_name",
    "product_id",
]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

for forbidden_feature in [
    "qty",
    "total_price",
    "calculated_revenue",
    "lag_price",
]:
    if forbidden_feature in FEATURES:
        raise ValueError(
            f"Leakage-risk feature included: {forbidden_feature}"
        )

X_train = train_df[FEATURES]
y_train = train_df[TARGET]
X_validation = validation_df[FEATURES]
y_validation = validation_df[TARGET]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Use dense output for compatibility with all selected estimators.
try:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    )
except TypeError:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False,
    )

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", encoder),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
])

print(f"Training rows: {len(train_df):,}")
print(f"Validation rows: {len(validation_df):,}")
print(f"Untouched test rows: {len(test_df):,}")


In [ ]:
def regression_metrics(actual, predicted):
    predicted_array = np.maximum(
        np.asarray(predicted, dtype=float),
        0,
    )
    return {
        "MAE": mean_absolute_error(actual, predicted_array),
        "RMSE": mean_squared_error(actual, predicted_array) ** 0.5,
        "R2": r2_score(actual, predicted_array),
    }


model_results = []
validation_predictions = {}

training_median_prediction = np.repeat(
    y_train.median(),
    len(y_validation),
)
validation_predictions["Training median baseline"] = (
    training_median_prediction
)
model_results.append({
    "Model": "Training median baseline",
    **regression_metrics(y_validation, training_median_prediction),
})

previous_quantity_prediction = (
    validation_df["previous_qty"]
    .fillna(y_train.median())
    .to_numpy()
)
validation_predictions["Previous quantity baseline"] = (
    previous_quantity_prediction
)
model_results.append({
    "Model": "Previous quantity baseline",
    **regression_metrics(y_validation, previous_quantity_prediction),
})

candidate_models = {
    "Ridge regression": Ridge(alpha=10.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=4,
        max_features=0.7,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Histogram Gradient Boosting": HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_iter=200,
        max_leaf_nodes=15,
        min_samples_leaf=12,
        l2_regularization=1.0,
        random_state=RANDOM_STATE,
    ),
}

fitted_models = {}

for model_name, estimator in candidate_models.items():
    model_pipeline = Pipeline([
        ("preprocess", clone(preprocessor)),
        ("model", estimator),
    ])
    model_pipeline.fit(X_train, y_train)
    prediction = np.maximum(
        model_pipeline.predict(X_validation),
        0,
    )
    fitted_models[model_name] = model_pipeline
    validation_predictions[model_name] = prediction
    model_results.append({
        "Model": model_name,
        **regression_metrics(y_validation, prediction),
    })

model_comparison = (
    pd.DataFrame(model_results)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

previous_baseline_rmse = model_comparison.loc[
    model_comparison["Model"] == "Previous quantity baseline",
    "RMSE",
].iloc[0]

model_comparison["RMSE_Improvement_vs_Previous_Qty_%"] = (
    (previous_baseline_rmse - model_comparison["RMSE"])
    / previous_baseline_rmse
    * 100
)

display(model_comparison)
model_comparison.to_csv(
    TABLES_DIR / "table_20_model_comparison.csv",
    index=False,
)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
plot_data = model_comparison.sort_values("RMSE", ascending=False)

sns.barplot(
    data=plot_data,
    x="RMSE",
    y="Model",
    color="#4C78A8",
    ax=ax,
)
ax.set_title("Validation RMSE by Preliminary Demand Model")
ax.set_xlabel("RMSE on March-May 2018 validation data")
ax.set_ylabel("Model")
fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "figure_09_model_comparison.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


In [ ]:
best_model_name = model_comparison.iloc[0]["Model"]
best_validation_prediction = validation_predictions[best_model_name]

prediction_table = validation_df[[
    "product_id",
    "product_category_name",
    "month_year",
    "qty",
    "unit_price",
]].copy()

prediction_table["Predicted_qty"] = best_validation_prediction
prediction_table["Residual"] = (
    prediction_table["qty"] - prediction_table["Predicted_qty"]
)
prediction_table["Model"] = best_model_name

prediction_table.to_csv(
    TABLES_DIR / "table_20b_validation_predictions.csv",
    index=False,
)

fig, ax = plt.subplots(figsize=(6.5, 6))
sns.scatterplot(
    data=prediction_table,
    x="qty",
    y="Predicted_qty",
    alpha=0.75,
    ax=ax,
)
plot_limit = max(
    prediction_table["qty"].max(),
    prediction_table["Predicted_qty"].max(),
)
ax.plot(
    [0, plot_limit],
    [0, plot_limit],
    color="#D62728",
    linestyle="--",
)
ax.set_title(f"Observed versus Predicted Quantity: {best_model_name}")
ax.set_xlabel("Observed quantity")
ax.set_ylabel("Predicted quantity")
fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "figure_10_preliminary_predictions.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## 8. Chronological learning curve


In [ ]:
months = sorted(pd.to_datetime(df["month_year"].unique()))
learning_curve_rows = []

reference_model = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", Ridge(alpha=10.0)),
])

for training_month_count in [6, 8, 10, 12, 14]:
    if training_month_count >= len(months):
        continue

    training_months = months[:training_month_count]
    validation_month = months[training_month_count]

    curve_train = df[df["month_year"].isin(training_months)].copy()
    curve_validation = df[df["month_year"] == validation_month].copy()

    if curve_train.empty or curve_validation.empty:
        continue

    curve_model = clone(reference_model)
    curve_model.fit(
        curve_train[FEATURES],
        curve_train[TARGET],
    )
    curve_prediction = np.maximum(
        curve_model.predict(curve_validation[FEATURES]),
        0,
    )

    curve_metrics = regression_metrics(
        curve_validation[TARGET],
        curve_prediction,
    )

    learning_curve_rows.append({
        "Training_Months": training_month_count,
        "Training_Rows": len(curve_train),
        "Validation_Month": pd.Timestamp(validation_month),
        **curve_metrics,
    })

learning_curve = pd.DataFrame(learning_curve_rows)

if learning_curve.empty:
    raise ValueError("The chronological learning curve produced no results.")

display(learning_curve)
learning_curve.to_csv(
    TABLES_DIR / "table_20c_learning_curve.csv",
    index=False,
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    learning_curve["Training_Rows"],
    learning_curve["RMSE"],
    marker="o",
)
ax.set_title("Chronological Learning Curve: Ridge Demand Model")
ax.set_xlabel("Cumulative training rows")
ax.set_ylabel("Next-month RMSE")
fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "figure_08_learning_curve.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## 9. Preliminary conclusions mapped to the research questions


In [ ]:
price_row = rq2_table.loc[
    rq2_table["Term"] == "log_unit_price"
].iloc[0]

research_question_summary = pd.DataFrame([
    {
        "Research_Question": "RQ1",
        "Preliminary_Conclusion": (
            f"Welch ANOVA p={welch_result.pvalue:.4g}; "
            f"descriptive eta-squared={eta_squared:.4f}."
        ),
        "Status": "Preliminary statistical result",
    },
    {
        "Research_Question": "RQ2",
        "Preliminary_Conclusion": (
            f"Conditional log-unit-price coefficient="
            f"{price_row['Coefficient']:.4f}, clustered p="
            f"{price_row['p_value']:.4g}. This is associational."
        ),
        "Status": "Preliminary association model",
    },
    {
        "Research_Question": "RQ3",
        "Preliminary_Conclusion": (
            f"Best validation model by RMSE: {best_model_name}. "
            "The test partition remains untouched."
        ),
        "Status": "Preliminary out-of-time validation",
    },
    {
        "Research_Question": "RQ4",
        "Preliminary_Conclusion": (
            f"The interaction is exploratory because {sparse_cells} "
            "category-position cells contain fewer than five observations."
        ),
        "Status": "Exploratory",
    },
])

display(research_question_summary)
research_question_summary.to_csv(
    TABLES_DIR / "table_22_research_question_summary.csv",
    index=False,
)


## 10. Completion checklist


In [ ]:
metadata = {
    "rows": int(len(df)),
    "products": int(df["product_id"].nunique()),
    "categories": int(df["product_category_name"].nunique()),
    "train_rows": int((df["data_partition"] == "Train").sum()),
    "validation_rows": int((df["data_partition"] == "Validation").sum()),
    "test_rows": int((df["data_partition"] == "Test").sum()),
    "best_validation_model": str(best_model_name),
    "test_set_used_for_model_selection": False,
    "random_state": RANDOM_STATE,
    "target": TARGET,
    "interpretation": (
        "Associational analysis; simulated revenue is not causal uplift or profit."
    ),
}

with open(
    PROCESSED_DIR / "notebook_03_metadata.json",
    "w",
    encoding="utf-8",
) as metadata_file:
    json.dump(metadata, metadata_file, indent=2)

expected_outputs = [
    TABLES_DIR / "table_17_sample_size_analysis.csv",
    TABLES_DIR / "table_18_rq1_statistical_results.csv",
    TABLES_DIR / "table_19_rq2_regression_results.csv",
    TABLES_DIR / "table_20_model_comparison.csv",
    TABLES_DIR / "table_21_rq4_interaction_results.csv",
    TABLES_DIR / "table_22_research_question_summary.csv",
    FIGURES_DIR / "figure_06_rq1_category_comparison.png",
    FIGURES_DIR / "figure_07_regression_diagnostics.png",
    FIGURES_DIR / "figure_08_learning_curve.png",
    FIGURES_DIR / "figure_09_model_comparison.png",
    FIGURES_DIR / "figure_10_preliminary_predictions.png",
    PROCESSED_DIR / "notebook_03_metadata.json",
]

completion_check = pd.DataFrame({
    "Output": [
        str(path.relative_to(PROJECT_ROOT))
        for path in expected_outputs
    ],
    "Exists": [path.exists() for path in expected_outputs],
})

display(completion_check)

if completion_check["Exists"].all():
    print(
        "Notebook 03 completed successfully. "
        "All expected outputs were created."
    )
else:
    missing_outputs = completion_check.loc[
        ~completion_check["Exists"],
        "Output",
    ].tolist()
    raise FileNotFoundError(
        "Missing expected outputs: " + ", ".join(missing_outputs)
    )


## 11. Next step

After successful execution:

1. Save this notebook.
2. Export it as `03_statistical_analysis_and_preliminary_models.html`.
3. Upload the HTML for verification.
4. Draft the interim report only from verified notebook outputs.
